# 00 · Preparar o ambiente
Cria o **esquema** e o **volume** da prática no Unity Catalog.

Rode as células em ordem (`Shift + Enter`) usando a computação **Serverless**.

In [ ]:
CATALOGO = "workspace"
ESQUEMA  = "aula_jobs"
VOLUME   = "arquivos"

BASE    = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUME}"
CHEGADA = f"{BASE}/chegada/"   # onde VOCÊ faz o upload dos 3 CSVs
PEDIDOS = f"{BASE}/pedidos/"   # onde o pipeline lê (o job copia um lote por vez)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUME}")
dbutils.fs.mkdirs(CHEGADA)
dbutils.fs.mkdirs(PEDIDOS)
print("Pronto! Volume criado em", BASE)

## Agora faça o upload dos lotes
1. Menu lateral **Catalog** → `workspace` → `aula_jobs` → **Volumes** → `arquivos`.
2. Entre na pasta **`chegada`**.
3. Clique em **Upload to this volume** e envie os 3 arquivos `pedidos_2026-10-1X.csv`.

> Não coloque nada na pasta `pedidos/` — ela é alimentada pelo job, um lote por vez.

In [ ]:
arquivos = sorted(f.name for f in dbutils.fs.ls(CHEGADA) if f.name.endswith(".csv"))
print(f"{len(arquivos)} arquivo(s) em chegada/:")
for a in arquivos:
    print("  -", a)

if len(arquivos) == 3:
    print("\n✅ Tudo certo. Siga para a Parte A do roteiro.")
else:
    print("\n⚠️ Esperava 3 CSVs. Refaça o upload ou use o Plano B abaixo.")

## Plano B — só se o upload não funcionar
A célula abaixo **gera os mesmos 3 lotes** direto na pasta `chegada/`. Rode-a apenas se a verificação acima não encontrou os arquivos.

In [ ]:
# Plano B: gera os lotes diretamente no volume
import csv
import os
import random

MUNICIPIOS = [
    ("Belém", 30), ("Ananindeua", 14), ("Abaetetuba", 10), ("Igarapé-Miri", 10),
    ("Cametá", 8), ("Marituba", 7), ("Castanhal", 7), ("Barcarena", 6),
    ("Breves", 4), ("Santarém", 4),
]
TIPOS = {"Açaí fino": 20.0, "Açaí médio": 28.0, "Açaí grosso": 38.0}
CANAIS = [("Batedeira", 55), ("Delivery", 30), ("Revenda", 15)]

# (data, total de linhas, semente, defeitos)
# defeitos: id_nulo / qtd_invalida / preco_invalido -> descartados na silver
#           municipio_vazio -> só gera alerta (warn), a linha segue
LOTES = [
    ("2026-10-14", 120, 14, {"id_nulo": 1, "qtd_invalida": 1, "preco_invalido": 1, "municipio_vazio": 2}),
    ("2026-10-15", 130, 15, {"id_nulo": 2, "qtd_invalida": 2, "preco_invalido": 1, "municipio_vazio": 2}),
    ("2026-10-16", 140, 16, {"id_nulo": 2, "qtd_invalida": 16, "preco_invalido": 3, "municipio_vazio": 3}),
]
COLUNAS = ["id_pedido", "data_pedido", "municipio", "tipo_acai", "canal",
           "qtd_litros", "preco_litro", "id_cliente"]


def _sorteia(rng, pares):
    nomes = [p[0] for p in pares]
    pesos = [p[1] for p in pares]
    return rng.choices(nomes, weights=pesos, k=1)[0]


def gerar_lote(data, n, semente, defeitos):
    rng = random.Random(semente)
    linhas = []
    for i in range(1, n + 1):
        canal = _sorteia(rng, CANAIS)
        tipo = rng.choice(list(TIPOS))
        if canal == "Revenda":
            qtd = float(rng.choice([10, 15, 20, 30, 50]))
            preco = TIPOS[tipo] * 0.85
        else:
            qtd = rng.choice([0.5, 1.0, 1.0, 1.0, 2.0, 2.0, 3.0, 5.0])
            preco = TIPOS[tipo]
        preco = round(preco * rng.uniform(0.92, 1.10), 2)
        linhas.append({
            "id_pedido": f"P{data.replace('-', '')}{i:04d}",
            "data_pedido": data,
            "municipio": _sorteia(rng, MUNICIPIOS),
            "tipo_acai": tipo,
            "canal": canal,
            "qtd_litros": f"{qtd:.1f}",
            "preco_litro": f"{preco:.2f}",
            "id_cliente": f"C{rng.randint(1, 400):04d}",
        })

    # Defeitos em posições sorteadas, sem repetir linha (contagens são fixas)
    total = sum(defeitos.values())
    posicoes = rng.sample(range(n), total)
    cursor = 0
    for tipo_defeito, qtd_def in defeitos.items():
        for _ in range(qtd_def):
            linha = linhas[posicoes[cursor]]
            cursor += 1
            if tipo_defeito == "id_nulo":
                linha["id_pedido"] = ""
            elif tipo_defeito == "qtd_invalida":
                # lote de 16/10: o PDV de Igarapé-Miri passou a mandar estornos negativos
                if qtd_def > 5:
                    linha["municipio"] = "Igarapé-Miri"
                    linha["qtd_litros"] = f"-{linha['qtd_litros']}"
                else:
                    linha["qtd_litros"] = "0.0" if cursor % 2 else f"-{linha['qtd_litros']}"
            elif tipo_defeito == "preco_invalido":
                linha["preco_litro"] = "" if cursor % 2 else "0.00"
            elif tipo_defeito == "municipio_vazio":
                linha["municipio"] = ""

    # Sujeira leve que a silver resolve com TRIM: espaços sobrando no município
    for j in rng.sample(range(n), 4):
        if linhas[j]["municipio"]:
            linhas[j]["municipio"] = linhas[j]["municipio"] + "  "
    return linhas


def gravar_lotes(pasta_destino):
    os.makedirs(pasta_destino, exist_ok=True)
    nomes = []
    for data, n, semente, defeitos in LOTES:
        nome = f"pedidos_{data}.csv"
        with open(os.path.join(pasta_destino, nome), "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=COLUNAS)
            w.writeheader()
            w.writerows(gerar_lote(data, n, semente, defeitos))
        nomes.append(nome)
    return nomes

print(gravar_lotes(CHEGADA))

## Recomeçar do zero (opcional)
Se precisar refazer a prática: descomente e rode a célula abaixo, depois rode o pipeline com **Run pipeline with full table refresh**.

In [ ]:
# for f in dbutils.fs.ls(PEDIDOS):
#     dbutils.fs.rm(f.path)
# print("Pasta pedidos/ esvaziada.")